# Análise de Insights — Aquecimento de Água no Brasil

> **Pergunta de decisão:** com capacidade de vendas limitada, **em quais UFs entrar primeiro, com qual produto e com qual discurso?**

Este notebook não repete a descrição das variáveis — ele **interpreta** os painéis já gerados e
fecha cada bloco com uma **recomendação**. Quatro achados:

1. O produto que funciona melhor e o cliente que paga estão em **regiões opostas**.
2. O consumo elétrico total **mede ar-condicionado, não aquecimento** — o proxy de demanda precisa mudar.
3. O Google Trends mostra **quando** a demanda por aquecimento acontece (inverno).
4. Ressalva: o índice `oportunidade_conversao` é **parcialmente circular** e não validado.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Resolve a pasta de dados rodando tanto localmente (repo) quanto no Colab (Drive)
CANDIDATOS = [
    "dados_projeto_ciencia_dados/",
    "/content/drive/MyDrive/projeto_ciencia_de_dados/dados_projeto_ciencia_dados/",
    "./",
]
BASE = next((p for p in CANDIDATOS if os.path.exists(p + "painel_uf_integrado.csv")), None)
assert BASE, "Nao encontrei os CSVs. Ajuste BASE para a pasta dados_projeto_ciencia_dados/."
print("Lendo de:", BASE)

integrado = pd.read_csv(BASE + "painel_uf_integrado.csv")
clusters  = pd.read_csv(BASE + "painel_uf_clusters.csv")[["uf", "segmento"]]
mensal    = pd.read_csv(BASE + "painel_mensal_uf.csv")
trends    = pd.read_csv(BASE + "gtrends_serie_temporal.csv")

uf = integrado.merge(clusters, on="uf", how="left")
mensal["regiao"] = mensal["uf"].map(uf.set_index("uf")["regiao"])
print("UFs:", uf.shape, "| mensal:", mensal.shape, "| trends:", trends.shape)

---
## Achado 1 — O fit do produto e a capacidade de pagar estão invertidos

Médias por segmento das variáveis que importam para a decisão comercial.

In [ ]:
cols = ["radiacao_media", "cop_medio_anual", "aptidao_solar",
        "rendimento_medio_pc", "temp_minima_inverno", "oportunidade_conversao",
        "trend_chuveiro_eletrico"]
resumo = uf.groupby("segmento")[cols].mean().round(2)
resumo = resumo.reindex(["Prêmio premium", "Vitrine solar", "Fronteira morna", "Baixa prioridade"])
display(resumo)

# A inversao em uma frase numerica:
c = uf.select_dtypes("number").corr()["oportunidade_conversao"]
print()
print(f"oportunidade_conversao x COP        : {c['cop_medio_anual']:+.2f}")
print(f"oportunidade_conversao x aptidao_solar: {c['aptidao_solar']:+.2f}")
print("=> a 'melhor oportunidade' cai onde os produtos rendem mais (sol/COP).")

**Mapa estratégico (2×2):** capacidade de pagar (renda) × aptidão solar. O tamanho da bolha é a `oportunidade_conversao`; a cor é o segmento.

In [ ]:
CORES = {"Prêmio premium": "#663399", "Vitrine solar": "#FF8C00",
         "Fronteira morna": "#2E8B57", "Baixa prioridade": "#808080"}
x, y = "rendimento_medio_pc", "aptidao_solar"
mx, my = uf[x].median(), uf[y].median()

fig, ax = plt.subplots(figsize=(10, 7))
for seg, g in uf.groupby("segmento"):
    ax.scatter(g[x], g[y], s=g["oportunidade_conversao"] * 8,
               c=CORES.get(seg, "#999"), alpha=0.75, edgecolor="white", label=seg)
for _, r in uf.iterrows():
    ax.annotate(r["uf"], (r[x], r[y]), fontsize=8, ha="center", va="center")
ax.axvline(mx, color="gray", ls="--", lw=1); ax.axhline(my, color="gray", ls="--", lw=1)

bbox = dict(boxstyle="round", fc="#f5f5f5", ec="gray", alpha=0.8)
ax.text(0.98, 0.97, "PAGA + APTO\n(raro: poucos aqui)", transform=ax.transAxes, ha="right", va="top", fontsize=9, bbox=bbox)
ax.text(0.98, 0.03, "PAGA, sol fraco (S/SE)\n→ premium: conforto/backup", transform=ax.transAxes, ha="right", va="bottom", fontsize=9, bbox=bbox)
ax.text(0.02, 0.97, "APTO, renda baixa (NE)\n→ solar c/ financiamento", transform=ax.transAxes, ha="left", va="top", fontsize=9, bbox=bbox)
ax.text(0.02, 0.03, "Baixa prioridade", transform=ax.transAxes, ha="left", va="bottom", fontsize=9, bbox=bbox)

ax.set_xlabel("Renda média per capita (R$)"); ax.set_ylabel("Aptidão solar")
ax.set_title("Mapa estratégico: quem paga × onde o solar rende")
ax.legend(title="Segmento", loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=4, fontsize=8)
plt.tight_layout(); plt.show()

> **Recomendação (Achado 1):** trate como **dois mercados distintos**, não um ranking único. No **Nordeste** (alta aptidão, renda baixa) o gargalo é financeiro → venda **solar com financiamento/payback**. No **Sul/Sudeste** (renda alta, frio, sol fraco e COP baixo) → venda **substituição premium do chuveiro** (conforto e custo de operação), não bomba de calor pura. O quadrante 'paga + apto' é pequeno — por isso não existe uma 'lista única de melhores UFs'.

---
## Achado 2 — O consumo elétrico total mede ar-condicionado, não aquecimento

Se o consumo residencial fosse dirigido por aquecimento de água, ele **subiria no frio**. Ele faz o contrário.

In [ ]:
# Correlacao intra-UF entre temperatura e consumo (positiva = dirigido por refrigeracao)
corr_uf = mensal.groupby("uf").apply(
    lambda g: g["temp_media_mensal"].corr(g["consumo_residencial_mwh"]), include_groups=False)
por_regiao = corr_uf.groupby(mensal.groupby("uf")["regiao"].first()).median().sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
por_regiao.plot.barh(ax=ax, color="#c0392b")
ax.axvline(0, color="k", lw=0.8)
ax.set_title("Correlação intra-UF: temperatura × consumo residencial (mediana por região)")
ax.set_xlabel("correlação (positiva ⇒ consumo sobe no calor ⇒ ar-condicionado)")
plt.tight_layout(); plt.show()

In [ ]:
# Graus-dia de aquecimento (HDD): energia necessaria para aquecer, base 20 C (escolha de sensibilidade)
BASE_HDD = 20.0
mensal["hdd"] = np.clip(BASE_HDD - mensal["temp_media_mensal"], 0, None) * mensal["n_dias"]

# Tres "formas" sazonais normalizadas (media=100) para comparar o que cada proxy diz
def idx100(s): return 100 * s / s.mean()
consumo_m = idx100(mensal.groupby("mes")["consumo_residencial_mwh"].sum())
hdd_m     = idx100(mensal.groupby("mes")["hdd"].sum())
trd_m     = idx100(trends[trends.termo == "aquecedor solar"].groupby("mes")["indice_busca"].mean())

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(consumo_m.index, consumo_m.values, "-o", label="Consumo total (MWh)", color="#2980b9")
ax.plot(hdd_m.index, hdd_m.values, "-s", label="Necessidade de aquecer (graus-dia)", color="#c0392b")
ax.plot(trd_m.index, trd_m.values, "-^", label="Busca 'aquecedor solar' (Trends)", color="#FF8C00")
ax.axvspan(5, 8, color="gray", alpha=0.12)  # inverno
ax.text(6.5, ax.get_ylim()[1]*0.97, "inverno", ha="center", va="top", color="gray")
ax.set_xticks(range(1, 13)); ax.set_xlabel("Mês"); ax.set_ylabel("Índice (média do ano = 100)")
ax.set_title("Os proxies discordam: consumo pica no VERÃO; demanda real de aquecimento, no INVERNO")
ax.legend(); plt.tight_layout(); plt.show()

print("Pico consumo:", consumo_m.idxmax(), "| Pico graus-dia:", hdd_m.idxmax(), "| Pico Trends:", trd_m.idxmax())

In [ ]:
# A necessidade de aquecimento (HDD anual) e hiperconcentrada no Sul
hdd_uf = mensal.groupby("uf")["hdd"].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
hdd_uf.head(12).plot.bar(ax=ax, color="#c0392b")
ax.set_title("Graus-dia de aquecimento por UF (base 20°C) — top 12")
ax.set_ylabel("graus-dia · ano"); plt.tight_layout(); plt.show()
print("Top 5:", hdd_uf.head(5).round(0).to_dict())

> **Recomendação (Achado 2):** **pare de usar MWh total como sinal de aquecimento** — ele segue o ar-condicionado. Use **graus-dia** como proxy físico da necessidade. Eles mostram que a necessidade de aquecimento de água é **dominada por RS, SC e PR** (uma ordem de grandeza acima do resto). Isso afina a 'capacidade de vendas limitada': o foco geográfico do produto de aquecimento é o **Sul**.

---
## Achado 3 — Quando vender: a sazonalidade está no Trends

O interesse de busca por todas as soluções de aquecimento **pica no inverno (mai–jul)** — o calendário comercial.

In [ ]:
piv = trends.groupby(["mes", "termo"])["indice_busca"].mean().unstack("termo")
fig, ax = plt.subplots(figsize=(10, 5))
for termo in piv.columns:
    ax.plot(piv.index, piv[termo], "-o", label=termo)
ax.axvspan(5, 8, color="gray", alpha=0.12)
ax.set_xticks(range(1, 13)); ax.set_xlabel("Mês"); ax.set_ylabel("Índice de busca (0–100)")
ax.set_title("Google Trends — sazonalidade do interesse por termo")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print("Mês de pico por termo:")
print(piv.idxmax())

> **Recomendação (Achado 3):** concentre campanhas e estoque **entre abril e julho**, antes/durante o pico de busca. "Chuveiro elétrico" domina o volume (mercado a substituir) e pica em maio; "aquecedor solar" pica em julho. Mensagem do Sul: "troque o chuveiro antes do inverno".

---
## Achado 4 (ressalva metodológica) — o índice de oportunidade é parcialmente circular

Antes de apresentar `oportunidade_conversao` como resultado, é preciso reconhecer como ele se forma.

In [ ]:
c = uf.select_dtypes("number").corr()["oportunidade_conversao"].drop("oportunidade_conversao")
c = c.reindex(c.abs().sort_values(ascending=False).index).head(12)
fig, ax = plt.subplots(figsize=(8, 6))
c.iloc[::-1].plot.barh(ax=ax, color=["#2E8B57" if v > 0 else "#c0392b" for v in c.iloc[::-1]])
ax.axvline(0, color="k", lw=0.8)
ax.set_title("Com o que 'oportunidade_conversao' se correlaciona")
ax.set_xlabel("correlação"); plt.tight_layout(); plt.show()

> **Ressalva (Achado 4):** o índice correlaciona **+0,84 com renda** e **±0,8 com sinais do próprio Trends** — ou seja, é em boa parte uma recombinação de variáveis que já entraram nele (circularidade), e **não foi validado** contra nenhuma medida real de adoção. **Próximo passo:** validar contra a **geração distribuída fotovoltaica por UF (relatórios ANEEL que já estão no projeto)** — se a aptidão/oportunidade prevê adoção real, o índice ganha credibilidade; se não, vira hipótese a investigar. Reporte-o como **índice de decisão com pesos explícitos**, não como achado.

---
## Síntese — uma decisão por segmento

| Mercado | Evidência | Produto | Discurso | Quando |
|---|---|---|---|---|
| **Sul (RS, SC, PR)** | Graus-dia 5–25× o resto; renda alta; frio | Substituição premium do chuveiro / solar c/ backup | Conforto + custo de operação no inverno | Campanha abr–jul |
| **Sudeste (SP, RJ, MG, ES)** | Renda alta, frio moderado | Solar + apoio elétrico | Economia na conta + conforto | abr–jul |
| **Nordeste (BA, PE, CE…)** | Melhor radiação, mas renda baixa | Solar de baixo custo | Financiamento / payment / payback rápido | ano todo (estável) |
| **Norte** | Baixa necessidade e renda | — | Despriorizar | — |

### Limitações a declarar no relatório
- **n = 27 UFs** → correlações são frágeis; tratar como indicativas, não conclusivas.
- **K-means (silhueta ≈ 0,34)** praticamente redescobre região + clima + renda; é uma rotulagem útil, não uma descoberta.
- **Granularidade UF** esconde heterogeneidade intraestadual (litoral × serra).
- **COP e aptidão solar são proxies** derivados de temperatura/radiação, não medições de engenharia.
- O proxy de demanda foi corrigido (graus-dia), mas **carece de validação** com dado de adoção real (ANEEL GD).